# 🚗 Training YOLOv11 — Vehicle CCTV Detection
### Jalankan cell satu per satu dari atas ke bawah!

**Struktur folder yang dibutuhkan:**
```
project_cctv/
├── train_yolov11.ipynb  ← file ini
└── vehicle-cctv-detection-1minutes_v1i_yolov11.zip  ← dataset zip
```

## Cell 1 — Cek GPU

In [ ]:
import torch

print('='*40)
print('CEK GPU')
print('='*40)
print(f'PyTorch version : {torch.__version__}')
print(f'CUDA available  : {torch.cuda.is_available()}')

if torch.cuda.is_available():
    print(f'GPU             : {torch.cuda.get_device_name(0)}')
    print(f'VRAM            : {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB')
    print('\n✅ GPU siap digunakan untuk training!')
else:
    print('\n⚠️ GPU tidak terdeteksi, training akan pakai CPU (lambat)')

## Cell 2 — Install Library

In [ ]:
# Install ultralytics
import subprocess
subprocess.run(['pip', 'install', 'ultralytics', '-q'])
print('✅ Ultralytics terinstall!')

## Cell 3 — Extract Dataset ZIP

In [ ]:
import zipfile
import os

# ── Sesuaikan nama ZIP kamu di sini ──────────
ZIP_PATH    = 'vehicle-cctv-detection-1minutes_v1i_yolov11.zip'
EXTRACT_DIR = 'dataset'
# ─────────────────────────────────────────────

if not os.path.exists(ZIP_PATH):
    print(f'[ERROR] File ZIP tidak ditemukan: {ZIP_PATH}')
    print('Pastikan file ZIP ada di folder yang sama dengan notebook ini!')
else:
    print(f'[INFO] Mengekstrak {ZIP_PATH}...')
    with zipfile.ZipFile(ZIP_PATH, 'r') as zip_ref:
        zip_ref.extractall(EXTRACT_DIR)
    print(f'✅ Dataset berhasil diekstrak ke folder: {EXTRACT_DIR}/')
    
    # Tampilkan isi folder
    print('\nIsi folder dataset:')
    for item in os.listdir(EXTRACT_DIR):
        print(f'  {item}')

## Cell 4 — Cek data.yaml

In [ ]:
import yaml
import os

# Cari file data.yaml di dalam folder dataset
yaml_path = None
for root, dirs, files in os.walk('dataset'):
    for file in files:
        if file == 'data.yaml':
            yaml_path = os.path.join(root, file)
            break

if yaml_path:
    print(f'✅ data.yaml ditemukan: {yaml_path}')
    with open(yaml_path, 'r') as f:
        data = yaml.safe_load(f)
    print('\nIsi data.yaml:')
    print(f'  Classes : {data.get("names", [])}')
    print(f'  nc      : {data.get("nc", 0)}')
else:
    print('[ERROR] data.yaml tidak ditemukan!')

## Cell 5 — Training YOLOv11 🚀

In [ ]:
from ultralytics import YOLO

# Load model YOLOv11 Nano (pretrained COCO)
model = YOLO('yolo11n.pt')

print('='*40)
print('MULAI TRAINING')
print('='*40)

# Training
results = model.train(
    data=yaml_path,      # path data.yaml
    epochs=50,           # jumlah epoch
    imgsz=640,           # ukuran gambar
    batch=16,            # batch size (kurangi ke 8 kalau VRAM kurang)
    device=0,            # 0 = GPU pertama
    project='runs',      # folder output
    name='cctv_vehicle', # nama experiment
    patience=10,         # early stopping
    verbose=True
)

print('\n✅ Training selesai!')

## Cell 6 — Cek Hasil & Lokasi best.pt

In [ ]:
import os

# Cari best.pt
best_pt = 'runs/cctv_vehicle/weights/best.pt'

if os.path.exists(best_pt):
    size = os.path.getsize(best_pt) / 1024**2
    print('='*40)
    print('TRAINING BERHASIL!')
    print('='*40)
    print(f'✅ best.pt ditemukan!')
    print(f'   Lokasi : {best_pt}')
    print(f'   Ukuran : {size:.1f} MB')
    print(f'\n📋 Langkah selanjutnya:')
    print(f'   1. Copy best.pt ke folder project kamu')
    print(f'   2. Ganti best.pt yang lama')
    print(f'   3. Jalankan: streamlit run capstonetester_app.py')
else:
    print('[ERROR] best.pt tidak ditemukan, cek output training di atas')

## Cell 7 — Validasi Model (Opsional)

In [ ]:
from ultralytics import YOLO

# Load best.pt dan validasi
model = YOLO('runs/cctv_vehicle/weights/best.pt')
metrics = model.val()

print('\n' + '='*40)
print('HASIL VALIDASI')
print('='*40)
print(f'mAP@50   : {metrics.box.map50:.3f}')
print(f'mAP@50-95: {metrics.box.map:.3f}')
print(f'Precision: {metrics.box.mp:.3f}')
print(f'Recall   : {metrics.box.mr:.3f}')